In [111]:
import numpy as np
import sys
import os
from math import ceil, log10

# Thêm thư mục cha (L02) vào sys.path
parent_path = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if parent_path not in sys.path:
    sys.path.append(parent_path)

from src.data_processing import load_csv#, one_hot_encode

Đọc dữ liệu gốc từ file data/raw/BankChurner.csv

In [112]:
DATA_PATH = "../data/raw/BankChurners.csv"
OUT_DIR = "../data/processed"
data, header = load_csv(DATA_PATH)

Loại bỏ các dấu nháy kép ( " )

In [113]:
def strip_quotes(s):
    return s.strip().strip('"').strip() if isinstance(s, str) else s

v_strip = np.vectorize(strip_quotes)
header = list(v_strip(header))
data = v_strip(data)

Ép kiểu các giá trị số (numeric value)

In [114]:
def convert_numeric(x):
    if not isinstance(x, str):
        return x
    
    try:
        # 1. Thử ép kiểu INT (số nguyên)
        # Không phải '1.335' hay '9.3e-05'
        if '.' not in x and 'e' not in x.lower():
            return int(x)
    except (ValueError, TypeError):
        pass 
    
    try:
        # 2. Thử ép kiểu FLOAT (số thực)
        # Ép kiểu '1.335' và '9.3448e-05'
        return float(x)
    except (ValueError, TypeError):
        # 3. Nếu không phải dạng số
        # Trả về chuỗi gốc
        return x

# test = data[:,0].astype(int)
v_convert_numeric = np.vectorize(convert_numeric, otypes=[object])
temp = v_convert_numeric(data)

Xử lý mã hóa các dữ liệu phân loại (feature encoding)

In [ ]:
cat_col_names = ['Gender', 'Education_Level', 'Marital_Status', 'Income_Category', 'Card_Category', 'Attrition_Flag']
cat_col_idx = [header.index(c) for c in cat_col_names]
num_col_names = [
    'Customer_Age', 'Dependent_count', 'Months_on_book', 
    'Total_Relationship_Count', 'Months_Inactive_12_mon', 'Contacts_Count_12_mon', 
    'Credit_Limit', 'Total_Revolving_Bal', 'Avg_Open_To_Buy', 
    'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Amt', 'Total_Trans_Ct', 
    'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio'
]
num_col_idx = [header.index(c) for c in num_col_names]

# Attrition_Flag
Attrition_idx = header.index('Attrition_Flag')
data[:, Attrition_idx] = np.where(data[:, Attrition_idx] == 'Attrited Customer', 1, 0)

# Gender
Gender_idx = header.index('Gender')
data[:, Gender_idx] = np.where(data[:, Gender_idx] == 'M', 1, 0)

# Education level
# Từ điển lưu giá trị số sẽ mã hóa
Edu_idx = header.index('Education_Level')
Edu_map = {'Unknown': 0, 'Uneducated': 1, 'High School': 2, 'College': 3, 'Graduate': 4, 'Post-Graduate': 5, 'Doctorate': 6}
Edu_keys = np.array(list(Edu_map.keys()))
Edu_values = np.array(list(Edu_map.values()))
sort_idx = np.argsort(Edu_keys)
mapped = Edu_values[sort_idx]
Edu_uni, inv = np.unique(data[:, Edu_idx], return_inverse=True)
# Sử dụng fancy indexing bằng mảng inv
data[:, Edu_idx] = mapped[inv]

# Income Category
inc_idx = header.index('Income_Category')
inc_map = {'Unknown': 0, 'Less than $40K': 1, '$40K - $60K': 2, '$60K - $80K': 3, '$80K - $120K': 4, '$120K +': 5}
inc_keys = np.array(list(inc_map.keys()))
inc_values = np.array(list(inc_map.values()))
sort_idx = np.argsort(inc_keys)
mapped = inc_values[sort_idx]
inc_uni, inv = np.unique(data[:, inc_idx], return_inverse=True)
# Sử dụng fancy indexing bằng mảng inv
data[:, inc_idx] = mapped[inv]

def ohe_numpy(col):
    unique_vals = np.unique(col)
    ohe_matrix = np.zeros((len(col), len(unique_vals)))
    for i, val in enumerate(unique_vals):
        ohe_matrix[col == val, i] = 1
    return ohe_matrix

def ohe_numpy_vectorized_2(col):
    unique_vals = np.unique(col)
    
    col_reshaped = col.reshape(-1, 1) # shape (n, 1)
    # 3. Dùng broadcasting để so sánh (n, 1) với (k,)
    # NumPy sẽ so sánh mỗi phần tử trong 'col_reshaped'
    # với TẤT CẢ các phần tử trong 'unique_vals'.
    # Kết quả là một ma trận Boolean (True/False) kích thước (n, k)
    boolean_matrix = (col_reshaped == unique_vals)
    
    # 4. Chuyển ma trận Boolean (True/False) thành số (1/0)
    return boolean_matrix.astype(int), unique_vals

# Marital Status
mari_idx = header.index('Marital_Status')

# Card Category

(10127, 1)
